# WebSight → формат контракта (drafting)

Конвертер: берёт срез **WebSight** и складывает его в датасет по **контракту `SFT/DATA_FORMAT_CONTRACT.md`** (схема §2, `save_to_disk`, картинки как `Image()`-байты), готовый к `load_from_disk` на SFT-стороне. Задача — `task_type="drafting"` (скриншот → HTML).

**Осознанно отложено (не блокирует пилот):**
- **Серые плейсхолдеры** (`APPLY_PLACEHOLDERS=False`). Контракт §3.1 разрешает «WebSight как есть». Включать плейсхолдеры в HTML можно только вместе с **ре-рендером** скриншота из placeholder-версии — иначе исходный скриншот с реальными картинками и код разойдутся. Ре-рендер = Этап 2 (рендерер eval-трека), его пока нет.
- **Фильтр по бюджету токенов** — сейчас грубо по символам (`MAX_HTML_CHARS`). Точный фильтр тем же токенайзером Qwen — когда SFT-трек его зафиксирует.
- **Tailwind precompile, декотаминация** — следующая итерация.

Цель шага — сквозной пайплайн end-to-end и первый LoRA у SFT-трека.

## 1. Импорты и конфиг

In [ ]:
import os, hashlib
from datasets import load_dataset, Dataset, Features, Image, Value, Sequence, load_from_disk
from bs4 import BeautifulSoup

# --- источник ---
DATASET = "mrm8488/WebSight_70k"   # лёгкий 70k-срез WebSight; поля: image (PIL), text (HTML)
                                   # полный: "HuggingFaceM4/WebSight" (~2M)
SPLIT   = "train"
N       = 300                      # размер пилотного среза (сотен хватит на первый LoRA)

# --- выход ---
OUT_DIR = os.path.abspath("./websight_drafting_pilot")

# --- флаги гигиены ---
APPLY_PLACEHOLDERS = False   # включать ТОЛЬКО вместе с ре-рендером (Этап 2) — см. заголовок
MAX_HTML_CHARS     = None    # напр. 8000; None = без фильтра. Точный фильтр по токенам — позже

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

print("out:", OUT_DIR)

## 2. Конвенция плейсхолдеров (идентична eval-треку)

In [ ]:
# ВАЖНО: ровно как в Evaluation/Experiments.ipynb. Менять только синхронно с eval.
PLACEHOLDER_CLASSES = ["bg-gray-300", "w-full", "h-48", "rounded"]
PLACEHOLDER_STYLE = "background-color:#d1d5db;width:100%;height:12rem;border-radius:0.5rem;display:block;"

def replace_images_with_placeholder(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    n = 0
    for img in soup.find_all("img"):
        div = soup.new_tag("div")
        div["class"] = PLACEHOLDER_CLASSES
        div["style"] = PLACEHOLDER_STYLE
        img.replace_with(div)
        n += 1
    return str(soup), n

## 3. Схема сэмпла (контракт §2)

In [ ]:
# Единая схема под 3 задачи; для drafting заполнены только images + target_html.
FEATURES = Features({
    "task_type":    Value("string"),
    "images":       Sequence(Image()),   # встроенные байты, НЕ пути к файлам
    "current_html": Value("string"),
    "target_html":  Value("string"),
    "instruction":  Value("string"),
})

## 4. Загрузка среза WebSight (streaming)

In [ ]:
# Стриминг — не тянем весь датасет, берём первые N примеров.
stream = load_dataset(DATASET, split=SPLIT, streaming=True)
raw = list(stream.take(N))
print(f"взято сырых примеров: {len(raw)}")
print("ключи примера:", list(raw[0].keys()))

## 5. Сборка сэмплов + базовая гигиена (дедуп, отсев пустых)

In [ ]:
rows, seen = [], set()
skipped_empty = skipped_dup = skipped_long = 0

for r in raw:
    html = (r.get("text") or r.get("html") or "").strip()
    img  = r.get("image")
    if not html or img is None:
        skipped_empty += 1
        continue
    if MAX_HTML_CHARS is not None and len(html) > MAX_HTML_CHARS:
        skipped_long += 1
        continue
    h = hashlib.sha1(html.encode("utf-8")).hexdigest()
    if h in seen:
        skipped_dup += 1
        continue
    seen.add(h)

    if APPLY_PLACEHOLDERS:
        html, _ = replace_images_with_placeholder(html)

    rows.append({
        "task_type":    "drafting",
        "images":       [img.convert("RGB")],
        "current_html": "",
        "target_html":  html,
        "instruction":  "",
    })

print(f"готово сэмплов: {len(rows)} | пропущено: пустых={skipped_empty}, дублей={skipped_dup}, длинных={skipped_long}")

## 6. Сохранение на диск

In [ ]:
ds = Dataset.from_list(rows, features=FEATURES)
ds.save_to_disk(OUT_DIR)
print("сохранено:", OUT_DIR)

## 7. Приёмка (контракт §6)

In [ ]:
ds2 = load_from_disk(OUT_DIR)
assert ds2.column_names == ["task_type","images","current_html","target_html","instruction"], ds2.column_names
s = ds2[0]
assert s["task_type"] == "drafting"
assert len(s["images"]) == 1 and s["images"][0].size
assert s["target_html"] and s["current_html"] == "" and s["instruction"] == ""
print("rows:", len(ds2))
print("image size:", s["images"][0].size, "| type:", type(s["images"][0]).__name__)
print("target_html head:", s["target_html"][:80])
print("OK — соответствует контракту (drafting)")
s["images"][0]   # покажет скриншот прямо в ноутбуке

## Готово / следующая итерация

**Приёмка пилота (контракт §6):** ✅ грузится `load_from_disk`, поля по §2, картинки `Image()`, сэмплы проверены глазами (последняя ячейка рисует скриншот).

**Отдать SFT-треку:** путь `OUT_DIR` смонтировать томом (контракт §7) → они читают `load_from_disk`.

**Докрутить, когда разблокируется:**
1. `APPLY_PLACEHOLDERS=True` + ре-рендер скриншотов (после Этапа 2 / рендерера eval).
2. Точный фильтр длины токенайзером Qwen (после согласования модели с SFT).
3. Tailwind precompile, дедуп по перцептивному хешу картинки, декотаминация доменов.
4. Масштаб: поднять `N` или перейти на полный `HuggingFaceM4/WebSight`.